# PixelTrace: reproduce the results on Colab

Runtime → Change runtime type → **GPU**. Then run the cells top to bottom.

* `MODE = "smoke"` runs the whole pipeline in minutes (1 epoch, 64 images per split) to check the setup.
  Smoke outputs go to `results/smoke/` and are never mixed with real results.
* `MODE = "full"` trains Simple CNN, ResNet-50 and EfficientNet-B0 on the fixed split, then runs
  robustness, Grad-CAM, model comparison, the validation-based deployment choice, ONNX export with
  parity checks, and a verification step that fails unless every artifact is present and consistent.

Any failing command stops the notebook with an error. Full mode requires Google Drive: it checks the backup folder is writable, copies the best checkpoint on
every validation improvement and each finished model immediately; a rerun restores finished models.

In [ ]:
MODE = "smoke"            # "smoke" first, then "full"
REPO_URL = ""             # e.g. "https://github.com/devbyabir/pixeltrace.git"; leave empty if the code is already in /content/pixeltrace
BACKUP = "/content/drive/MyDrive/pixeltrace_backup"   # required for MODE="full"
RAW_GDRIVE_ID = ""        # private: Drive file id of the raw dataset zip, if you have access (see DATASET.md)

In [ ]:
import os, subprocess

def sh(cmd: str) -> None:
    """Run a shell command, stream its output, raise on a non-zero exit code."""
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"command failed (exit {p.returncode}): {cmd}")

if REPO_URL and not os.path.exists("/content/pixeltrace"):
    sh(f"git clone -q {REPO_URL} /content/pixeltrace")
os.chdir("/content/pixeltrace")
sh("pip install -q -r requirements.txt && pip install -q --no-deps -e .")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")   # full training stops if Drive is not mounted
os.makedirs(BACKUP, exist_ok=True)
os.environ["PIXELTRACE_RAW_GDRIVE_ID"] = RAW_GDRIVE_ID


In [ ]:
backup = f"--backup {BACKUP}"   # full mode verifies it is writable and stops otherwise
skip = "--skip-data" if os.path.exists("data/processed/splits.csv") else ""
sh(f"python scripts/colab_pipeline.py --mode {MODE} {backup} {skip}")

In [ ]:
from IPython.display import Image, Markdown, display
base = "results/smoke" if MODE == "smoke" else "results"
display(Markdown(open(f"{base}/summary.md").read()))
display(Image(f"{base}/summary.png", width=650))